# Silver Layer: Feature Engineering with Quality-Gated Writes

**Input:** Bronze Delta tables `bronze_fd001`, `bronze_fd003` (train + test rows with a `split` column) and `bronze_rul_fd001`, `bronze_rul_fd003` (true RUL per test engine, keyed by `unit_id`).
**Output:** `silver_fd001` (33,727 rows) and `silver_fd003` (41,316 rows). FD001 is used for modeling; FD003 is prepared for future work.

## Engine identity
CMAPSS train and test files both number engines 1–100, but train engine 7 and test engine 7 are **different physical engines**. Every per-engine operation (RUL, rolling windows, quality checks) is keyed on **(`split`, `unit_id`)**.

## Transformations (each justified in `02_eda`)
1. **RUL label.** Train engines run to failure: RUL = last cycle − current cycle. Test engines stop early: RUL = (last observed cycle − current cycle) + the true RUL from `RUL_FD00x.txt`.
2. **Cap at 125** (piecewise-linear target; early-life degradation is not observable).
3. **Drop** 8 zero-variance columns and `sensor_14` (r = 0.963 with `sensor_9`).
4. **Rolling features:** 30-cycle mean and std per sensor, computed within each engine.
5. **Min-max scaling** fit on the training split only; test values clipped to [0, 1].
6. **Alert label** `alert_30` = 1 if capped RUL ≤ 30.

## Quality gates (a failed check blocks the write)
- Nulls in key columns, RUL bounds [0, 125], scaled values in [0, 1], binary alert label
- **Every training engine ends at RUL 0** (run to failure)
- **Every test engine's last-cycle RUL matches `RUL_FD00x.txt` exactly**

The last two checks were added after an earlier version keyed engines on `unit_id` alone, which merged train and test engines and corrupted test labels. They pass on the current data and fail on the old logic.

## Result (FD001)
| Split | Rows | Alert rows (RUL ≤ 30) |
|---|---|---|
| train | 20,631 | 3,100 (15.0%) |
| test | 13,096 | 332 (2.5%) |


In [0]:
# ============================================================
# CELL 1 — Silver Layer: Feature Engineering
# Notebook: 03_silver_layer
# Transforms: compute RUL (with test truth), drop zero-variance,
#             cap RUL, rolling windows, normalize, resolve
#             collinearity, write Silver Delta
#
# ENGINE KEY = (split, unit_id). CMAPSS train and test files both
# number engines 1-100, but train unit 7 and test unit 7 are
# DIFFERENT physical engines. Every per-engine window must
# partition on both columns.
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# --- Load Bronze tables ---
bronze_fd001 = spark.table("workspace.predictive_maintenance.bronze_fd001")
bronze_fd003 = spark.table("workspace.predictive_maintenance.bronze_fd003")
rul_fd001    = spark.table("workspace.predictive_maintenance.bronze_rul_fd001")
rul_fd003    = spark.table("workspace.predictive_maintenance.bronze_rul_fd003")

# --- EDA-driven constants ---
RUL_CAP = 125
ENGINE_KEY = ["split", "unit_id"]

FEATURES_TO_DROP = [
    "sensor_1", "sensor_5", "sensor_6", "sensor_10",
    "sensor_16", "sensor_18", "sensor_19", "op_setting_3"
]

# Drop sensor_14 to resolve 0.963 collinearity with sensor_9
COLLINEAR_DROP = ["sensor_14"]

ROLLING_WINDOW = 30  # cycles — gives Ridge temporal context

INFORMATIVE_SENSORS = [
    "sensor_2", "sensor_3", "sensor_4", "sensor_7", "sensor_8",
    "sensor_9", "sensor_11", "sensor_12", "sensor_13",
    "sensor_15", "sensor_17", "sensor_20", "sensor_21"
]

def build_silver(df, rul_truth_df, subset_label):
    """
    Apply all EDA-driven transformations to produce Silver features.
    Steps:
      1. Compute RUL per engine (split, unit_id):
           train (run to failure):  RUL = max_cycle - t
           test  (stops early):     RUL = (max_cycle - t) + rul_true from RUL_FD00x.txt
      2. Cap RUL at 125
      3. Drop zero-variance and collinear features
      4. Rolling mean + std per sensor (window=30) WITHIN each engine
      5. Min-max normalize sensor values (fit on train split only)
      6. Binary alert label
    """
    # Steps 1 & 2: Compute and cap RUL
    w_engine = Window.partitionBy(*ENGINE_KEY)
    truth = (rul_truth_df
             .select("unit_id", "rul_true")
             .withColumn("split", F.lit("test")))

    df = (df
          .withColumn("max_cycle", F.max("time_cycles").over(w_engine))
          .join(truth, on=["split", "unit_id"], how="left")
          .withColumn("RUL",
                      F.col("max_cycle") - F.col("time_cycles")
                      + F.coalesce(F.col("rul_true"), F.lit(0)))
          .withColumn("RUL_capped", F.least(F.col("RUL"), F.lit(RUL_CAP)))
          .drop("max_cycle", "rul_true")
    )

    # Step 3: Drop zero-variance and collinear features
    drop_cols = FEATURES_TO_DROP + COLLINEAR_DROP
    df = df.drop(*drop_cols)

    # Step 4: Rolling mean and std within one physical engine
    w_roll = (Window.partitionBy(*ENGINE_KEY)
              .orderBy("time_cycles")
              .rowsBetween(-ROLLING_WINDOW + 1, 0))

    for s in INFORMATIVE_SENSORS:
        df = (df
              .withColumn(f"{s}_roll_mean", F.mean(s).over(w_roll))
              .withColumn(f"{s}_roll_std",  F.stddev(s).over(w_roll))
        )

    # Fill nulls in roll_std for each engine's first cycle
    roll_std_cols = [f"{s}_roll_std" for s in INFORMATIVE_SENSORS]
    df = df.fillna(0.0, subset=roll_std_cols)

    # Step 5: Min-max normalize original sensor columns
    # Compute min/max from training split only to avoid leakage
    train_df = df.filter(F.col("split") == "train")
    for s in INFORMATIVE_SENSORS:
        min_val = train_df.agg(F.min(s)).collect()[0][0]
        max_val = train_df.agg(F.max(s)).collect()[0][0]
        rng = max_val - min_val if max_val != min_val else 1.0
        df = df.withColumn(
            f"{s}_norm",
            (F.col(s) - F.lit(min_val)) / F.lit(rng)
        )

    # Step 6: Maintenance alert label (binary — failure within 30 cycles)
    df = df.withColumn(
        "alert_30",
        F.when(F.col("RUL_capped") <= 30, 1).otherwise(0)
    )

    return df.withColumn("silver_subset", F.lit(subset_label))

print("Building Silver FD001...")
silver_fd001 = build_silver(bronze_fd001, rul_fd001, "FD001")

print("Building Silver FD003...")
silver_fd003 = build_silver(bronze_fd003, rul_fd003, "FD003")

print("Done. Sample schema:")
silver_fd001.printSchema()

Building Silver FD001...
Building Silver FD003...
Done. Sample schema:
root
 |-- split: string (nullable = true)
 |-- unit_id: integer (nullable = true)
 |-- time_cycles: integer (nullable = true)
 |-- op_setting_1: double (nullable = true)
 |-- op_setting_2: double (nullable = true)
 |-- sensor_2: double (nullable = true)
 |-- sensor_3: double (nullable = true)
 |-- sensor_4: double (nullable = true)
 |-- sensor_7: double (nullable = true)
 |-- sensor_8: double (nullable = true)
 |-- sensor_9: double (nullable = true)
 |-- sensor_11: double (nullable = true)
 |-- sensor_12: double (nullable = true)
 |-- sensor_13: double (nullable = true)
 |-- sensor_15: double (nullable = true)
 |-- sensor_17: integer (nullable = true)
 |-- sensor_20: double (nullable = true)
 |-- sensor_21: double (nullable = true)
 |-- subset: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- RUL: long (nullable = true)
 |-- RUL_capped: long (nullable = false)
 |-- sensor_2_roll_mean: doub

In [0]:
# ============================================================
# CELL 2 — Quality Checks + Write Silver Delta Tables
# Clip normalized values to [0,1] (test values outside train range),
# run quality gates, block the write on any failure.
# ============================================================

from pyspark.sql import functions as F

# Cast sensor_17 to double
silver_fd001 = silver_fd001.withColumn("sensor_17", F.col("sensor_17").cast("double"))
silver_fd003 = silver_fd003.withColumn("sensor_17", F.col("sensor_17").cast("double"))

# Clip all _norm columns to [0, 1] — handles test set range violations
norm_cols = [c for c in silver_fd001.columns if c.endswith("_norm")]
for col in norm_cols:
    silver_fd001 = silver_fd001.withColumn(col, F.least(F.lit(1.0), F.greatest(F.lit(0.0), F.col(col))))
    silver_fd003 = silver_fd003.withColumn(col, F.least(F.lit(1.0), F.greatest(F.lit(0.0), F.col(col))))

print(f"Clipped {len(norm_cols)} normalized columns to [0, 1]")

# ============================================================
# DATA QUALITY CHECKS
# ============================================================

def run_quality_checks(df, rul_truth_df, name):
    errors = []

    key_cols = ["unit_id", "time_cycles", "RUL_capped", "sensor_11", "sensor_4", "sensor_12"]
    for col in key_cols:
        null_count = df.filter(F.col(col).isNull()).count()
        if null_count > 0:
            errors.append(f"NULL CHECK FAILED: {col} has {null_count} nulls")

    rul_violations = df.filter((F.col("RUL_capped") < 0) | (F.col("RUL_capped") > 125)).count()
    if rul_violations > 0:
        errors.append(f"RUL BOUNDS FAILED: {rul_violations} rows outside [0, 125]")

    uid_violations = df.filter(F.col("unit_id") <= 0).count()
    if uid_violations > 0:
        errors.append(f"UNIT_ID CHECK FAILED: {uid_violations} non-positive unit_ids")

    for col in norm_cols:
        violations = df.filter((F.col(col) < -0.01) | (F.col(col) > 1.01)).count()
        if violations > 0:
            errors.append(f"NORM BOUNDS FAILED: {col} has {violations} values outside [0,1]")

    alert_vals = df.select("alert_30").distinct().count()
    if alert_vals > 2:
        errors.append(f"ALERT LABEL FAILED: alert_30 has {alert_vals} distinct values (expected 2)")

    # --- NEW: engine identity checks (engine = split + unit_id) ---
    # 1. Every TRAIN engine runs to failure, so its last cycle must be RUL 0
    bad_train = (df.filter("split = 'train'").groupBy("unit_id")
                   .agg(F.min("RUL").alias("last_rul"))
                   .filter("last_rul != 0").count())
    if bad_train > 0:
        errors.append(f"ENGINE KEY FAILED: {bad_train} train engines don't end at RUL 0")

    # 2. Every TEST engine's last-cycle RUL must equal RUL_FD00x.txt exactly
    test_last = (df.filter("split = 'test'").groupBy("unit_id")
                   .agg(F.min("RUL").alias("last_rul")))
    mismatches = (test_last.join(rul_truth_df.select("unit_id", "rul_true"), "unit_id", "full_outer")
                    .filter("last_rul IS NULL OR rul_true IS NULL OR last_rul != rul_true")
                    .count())
    if mismatches > 0:
        errors.append(f"TEST TRUTH FAILED: {mismatches} test engines don't match the RUL truth file")

    print(f"\n=== DATA QUALITY REPORT: {name} ===")
    if errors:
        for e in errors:
            print(f"  ❌ {e}")
        raise ValueError(f"Data quality checks failed for {name}. Blocking Silver write.")
    else:
        row_count = df.count()
        print(f"  ✅ All checks passed | {row_count:,} rows | {len(df.columns)} columns")
        print(f"  ✅ All train engines end at RUL 0; all test engines match the truth file")

run_quality_checks(silver_fd001, rul_fd001, "silver_fd001")
run_quality_checks(silver_fd003, rul_fd003, "silver_fd003")

# ============================================================
# WRITE SILVER DELTA TABLES
# overwriteSchema: column order changed with the engine-key join
# ============================================================

(silver_fd001.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.predictive_maintenance.silver_fd001"))

(silver_fd003.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.predictive_maintenance.silver_fd003"))

print("\n✅ Silver Delta tables written:")
print("   - workspace.predictive_maintenance.silver_fd001")
print("   - workspace.predictive_maintenance.silver_fd003")

# Alert rate PER SPLIT (train engines run to failure; test engines are truncated)
for name, tbl in [("silver_fd001", "workspace.predictive_maintenance.silver_fd001"),
                  ("silver_fd003", "workspace.predictive_maintenance.silver_fd003")]:
    df = spark.table(tbl)
    print(f"\n{name}: {df.count():,} rows")
    (df.groupBy("split")
       .agg(F.count("*").alias("rows"),
            F.sum("alert_30").alias("alert_rows"),
            F.round(F.avg("alert_30") * 100, 1).alias("alert_pct"))
       .orderBy("split").show())

Clipped 13 normalized columns to [0, 1]

=== DATA QUALITY REPORT: silver_fd001 ===
  ✅ All checks passed | 33,727 rows | 63 columns
  ✅ All train engines end at RUL 0; all test engines match the truth file

=== DATA QUALITY REPORT: silver_fd003 ===
  ✅ All checks passed | 41,316 rows | 63 columns
  ✅ All train engines end at RUL 0; all test engines match the truth file

✅ Silver Delta tables written:
   - workspace.predictive_maintenance.silver_fd001
   - workspace.predictive_maintenance.silver_fd003

silver_fd001: 33,727 rows
+-----+-----+----------+---------+
|split| rows|alert_rows|alert_pct|
+-----+-----+----------+---------+
| test|13096|       332|      2.5|
|train|20631|      3100|     15.0|
+-----+-----+----------+---------+


silver_fd003: 41,316 rows
+-----+-----+----------+---------+
|split| rows|alert_rows|alert_pct|
+-----+-----+----------+---------+
| test|16596|       291|      1.8|
|train|24720|      3100|     12.5|
+-----+-----+----------+---------+

